# Clasificación inteligente de eventos de seguridad (IDS) con Machine Learning

**Semana 2 — Inteligencia Artificial**  
**Autor:** Alejandro Carriel · PUCE · Ingeniería / Ciberseguridad · 5.º semestre

Notebook académico: carga y exploración de un subconjunto de **CICIDS2017**, exclusión de columnas prohibidas, entrenamiento de tres modelos (Árbol de decisión, Random Forest y SVM) y evaluación orientada a IDS (énfasis en falsos negativos).


## 1. Importación de librerías


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    accuracy_score,
    classification_report,
    RocCurveDisplay,
)

print("pandas", pd.__version__)
print("numpy", np.__version__)


## 2. Carga de datos

Los CSV se encuentran en `../data/` (rutas relativas a la carpeta `notebooks/`).


In [ ]:
data_dir = Path("../data")
results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(data_dir / "CICIDS2017_train.csv", low_memory=False)
test = pd.read_csv(data_dir / "CICIDS2017_test.csv", low_memory=False)

print("Train:", train.shape)
print("Test:", test.shape)
print("Últimas columnas:", train.columns[-4:].tolist())


## 3. Exploración de la etiqueta binaria

Distribución de `label_binary` (0 = benigno, 1 = ataque) en entrenamiento y prueba.


In [ ]:
print("=== TRAIN ===")
print(train["label_binary"].value_counts(dropna=False))
print(train["label_binary"].value_counts(normalize=True).round(4))

print("\n=== TEST ===")
print(test["label_binary"].value_counts(dropna=False))
print(test["label_binary"].value_counts(normalize=True).round(4))

print("\nValores únicos train:", train["label_binary"].unique())
print("Valores únicos test :", test["label_binary"].unique())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

train["label_binary"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], title="Train — label_binary"
)
test["label_binary"].value_counts().sort_index().plot(
    kind="bar", ax=axes[1], title="Test — label_binary"
)

axes[0].set_xlabel("0 = benigno, 1 = ataque")
axes[1].set_xlabel("0 = benigno, 1 = ataque")
plt.tight_layout()
plt.show()


## 4. Exclusión de columnas prohibidas

Según el enunciado de la tarea, las siguientes columnas **no** deben usarse como predictores:

- `label_multiclass`
- `attack_family`
- `label_binary`
- `source_file`

`label_binary` se reserva como variable objetivo. `attack_family` y `label_multiclass` se emplearán solo en el análisis de falsos negativos.


In [ ]:
prohibidas = [
    "label_multiclass",
    "attack_family",
    "label_binary",
    "source_file",
]

predictores = [c for c in train.columns if c not in prohibidas]

print("Columnas excluidas:", prohibidas)
print("N de predictores:", len(predictores))
print(predictores)

X_train = train[predictores].copy()
X_test = test[predictores].copy()
y_train = train["label_binary"].astype(int)
y_test = test["label_binary"].astype(int)

print("\nX_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test :", X_test.shape, "| y_test :", y_test.shape)
print("¿Se coló alguna prohibida?", any(c in X_train.columns for c in prohibidas))


## 5. Estadísticas descriptivas

Se exportan estadísticas del conjunto de entrenamiento a `../results/estadisticas_descriptivas_train.csv`.


In [ ]:
print("Variables predictoras (", len(predictores), "):")
for i, col in enumerate(predictores, start=1):
    print(f"{i:02d}. {col}")

desc = X_train.describe().T
desc["nulos"] = X_train.isna().sum()
desc["dtype"] = X_train.dtypes.astype(str)

print("\n=== Extracto (primeras 12 features) ===")
print(desc[["count", "mean", "std", "min", "max", "nulos"]].head(12))

out_stats = results_dir / "estadisticas_descriptivas_train.csv"
desc.to_csv(out_stats)
print("\nTabla completa guardada en", out_stats.as_posix())


## 6. Control de calidad: nulos, infinitos y duplicados


In [ ]:
nulos_train = int(X_train.isna().sum().sum())
nulos_test = int(X_test.isna().sum().sum())

Xtr = X_train.apply(pd.to_numeric, errors="coerce")
Xte = X_test.apply(pd.to_numeric, errors="coerce")

inf_train = int(np.isinf(Xtr.to_numpy()).sum())
inf_test = int(np.isinf(Xte.to_numpy()).sum())

dup_train = int(train.duplicated().sum())
dup_test = int(test.duplicated().sum())

print("Nulos train / test     :", nulos_train, "/", nulos_test)
print("Infinitos train / test :", inf_train, "/", inf_test)
print("Duplicados train / test:", dup_train, "/", dup_test)


## 7. Entrenamiento de modelos

Se entrenan tres clasificadores solo con el conjunto de entrenamiento (`X_train`, `y_train`), con hiperparámetros fijos para reproducibilidad (`random_state=42`).


In [ ]:
arbol = DecisionTreeClassifier(
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=18,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

arbol.fit(X_train, y_train)
rf.fit(X_train, y_train)

print("Árbol de decisión entrenado")
print("Random Forest entrenado")


In [ ]:
svm_pipe = Pipeline(
    steps=[
        ("escalado", StandardScaler()),
        (
            "svm",
            SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=42,
            ),
        ),
    ]
)

svm_pipe.fit(X_train, y_train)
print("SVM (RBF) + StandardScaler entrenado")


## 8. Métricas de evaluación (clase positiva = ataque = 1)

| Métrica | Función | Significado en un IDS |
|---|---|---|
| Matriz de confusión | `confusion_matrix` | TN, FP, FN, TP |
| Precision | `precision_score` | De las alertas, cuántas eran realmente ataque |
| Recall | `recall_score` | De los ataques reales, cuántos se detectaron |
| F1-score | `f1_score` | Equilibrio entre precision y recall |
| ROC-AUC | `roc_auc_score` | Capacidad de ordenar flujos por probabilidad de ataque |
| Accuracy | `accuracy_score` | Acierto global; **no usar de forma aislada** |

Interpretación de la matriz:

- **TN**: benigno bien clasificado  
- **FP**: benigno alertado (falsa alarma)  
- **FN**: ataque clasificado como benigno (**el error más grave en un IDS**)  
- **TP**: ataque detectado  


In [ ]:
def evaluar(nombre, modelo, X_test, y_test):
    """Evalúa un clasificador binario en test e imprime métricas orientadas a IDS."""
    y_pred = modelo.predict(X_test)
    y_score = modelo.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    print("=" * 60)
    print(nombre)
    print("=" * 60)
    print("Matriz [filas=real, cols=predicho]")
    print(cm)
    print(f"TN={tn} FP={fp} FN={fn} TP={tp}")
    print(f"Accuracy        : {accuracy_score(y_test, y_pred):.4f} (no usarla sola)")
    print(f"Precision ataque: {precision_score(y_test, y_pred, pos_label=1):.4f}")
    print(f"Recall ataque   : {recall_score(y_test, y_pred, pos_label=1):.4f}")
    print(f"F1 ataque       : {f1_score(y_test, y_pred, pos_label=1):.4f}")
    print(f"ROC-AUC         : {roc_auc_score(y_test, y_score):.4f}")
    print(classification_report(y_test, y_pred, target_names=["Benigno", "Ataque"]))
    return y_pred, y_score, cm


pred_arbol, score_arbol, cm_arbol = evaluar("Árbol de decisión", arbol, X_test, y_test)
pred_rf, score_rf, cm_rf = evaluar("Random Forest", rf, X_test, y_test)
pred_svm, score_svm, cm_svm = evaluar("SVM + Pipeline", svm_pipe, X_test, y_test)


## 9. Curvas ROC en el conjunto de prueba


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, score_arbol, name="Árbol", ax=ax)
RocCurveDisplay.from_predictions(y_test, score_rf, name="Random Forest", ax=ax)
RocCurveDisplay.from_predictions(y_test, score_svm, name="SVM", ax=ax)
ax.set_title("Curvas ROC en test")
plt.tight_layout()
plt.show()


## 10. Análisis de falsos negativos

Un **falso negativo** es un flujo que era ataque (`label_binary = 1`) y el modelo predijo benigno (`0`). En un IDS es el error más crítico: el ataque no genera alerta.

Se analiza el **árbol de decisión** (típicamente el de menor número de FN en este experimento).  
`attack_family` y `label_multiclass` **no** se usaron para entrenar; aquí solo sirven para caracterizar qué tipos de ataque se escaparon.


In [ ]:
fn_mask = (y_test.to_numpy() == 1) & (pred_arbol == 0)

n_ataques = int((y_test == 1).sum())
n_fn = int(fn_mask.sum())

print("Modelo           : Árbol de decisión")
print("Ataques en test  :", n_ataques)
print("Falsos negativos :", n_fn)
print("Porcentaje de FN :", round(100 * n_fn / n_ataques, 2), "%")

print("\nFN por attack_family:")
print(test.loc[fn_mask, "attack_family"].value_counts())

print("\nFN por label_multiclass:")
print(test.loc[fn_mask, "label_multiclass"].value_counts())


In [ ]:
familias = test.loc[fn_mask, "attack_family"].value_counts()

familias.plot(
    kind="bar",
    color="firebrick",
    title="Falsos negativos por familia de ataque (Árbol de decisión)",
)
plt.ylabel("Número de flujos FN")
plt.xlabel("attack_family")
plt.tight_layout()
plt.show()


## 11. Conclusiones

1. Se construyó un pipeline reproducible de clasificación binaria IDS sobre un subconjunto académico de **CICIDS2017**, respetando la exclusión de columnas prohibidas.
2. Los tres modelos (Árbol, Random Forest y SVM con escalado) permiten comparar precisión, recall, F1 y ROC-AUC en test.
3. En el contexto de un IDS, conviene priorizar el **recall** y minimizar **falsos negativos**, aunque ello implique un coste en falsas alarmas (FP).
4. El análisis de FN por `attack_family` ayuda a identificar familias de ataque más difíciles de detectar y orientar mejoras futuras (ingeniería de características, calibración o umbrales).
5. Este trabajo es estrictamente **educativo y defensivo**; no debe emplearse con fines ofensivos o no autorizados.

**Cita del dataset:** Sharafaldin, I., Lashkari, A. H., & Ghorbani, A. A. (2018). *Toward Generating a New Intrusion Detection Dataset and Intrusion Traffic Characterization*. ICISSP. Canadian Institute for Cybersecurity — https://www.unb.ca/cic/datasets/ids-2017.html
